<a href="https://colab.research.google.com/github/Ecc535/Mental_Health_LLM_Benchmarking/blob/main/Mental_LLM_Benchmarking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U vllm
!pip uninstall -y protobuf
!pip install protobuf==4.25.3

In [1]:
import subprocess
import time
import os
import signal
import requests
from openai import OpenAI
from openai import APIConnectionError
import time
import json
import concurrent.futures
import threading
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

In [2]:
models = {
    "Llama-Nemotron-4B": {
        "hf_name": "nvidia/Llama-3.1-Nemotron-Nano-4B-v1.1",
        "port": 5000,
        "gpu_memory_utilization": 0.85,
        "max_model_len": 8192
    },
    "BioMistral-7B": {
        "hf_name": "BioMistral/BioMistral-7B",
        "port": 5001,
        "gpu_memory_utilization": 0.85,
        "max_model_len": 4096
    }
}

**Verify API server spin up**

In [3]:
def wait_for_server(log_file_path, process):
    print(f"Waiting for vLLM server to spin up. Monitoring {log_file_path}...")
    start_time = time.time()

    while True:
        # CHECK 1: Did the process crash while we were waiting?
        if process.poll() is not None:
            print(f"\n[ERROR] Server process died unexpectedly (Code {process.returncode}).")
            if os.path.exists(log_file_path):
                with open(log_file_path, "r") as f:
                    lines = f.readlines()
                    print(f"--- Last 10 lines of log ---\n{''.join(lines[-10:])}")
            return False

        # CHECK 2: Is the startup complete?
        if os.path.exists(log_file_path):
            with open(log_file_path, "r") as f:
                if "Application startup complete" in f.read():
                    elapsed = time.time() - start_time
                    print(f"\n[SUCCESS] Server is up and running! (Took {elapsed:.1f} seconds)")
                    return True

        print(".", end="", flush=True)
        time.sleep(5)

In [4]:
import pandas as pd

In [5]:
FEATURE_GROUPS = {
    "Rhythm": [
        "steps_48phi", "steps_8phi", "steps_psd_auc_64h",
        "steps_12p_reject", "steps_128p_reject", "steps_IV"
    ],
    "Heart Rate Variability": [
        "hrstd_max", "hrstd_std", "hrmax_std", "hrmean_max",
        "hrmax_mean", "hrentropy_std", "RestingHeartRate_std", "hrmax_max"
    ],
    "Shift Pattern": [
        "shift_hours_median", "shift_type_shannon_entropy",
        "shift_hours_shannon_entropy"
    ],
    "Inactivity at Daytime": [
        "duration_entropy_non_step_std"
    ],
    "Morning Energy": [
        "happiness_morning_std", "happiness_morning_shannon_entropy",
        "sleepiness_daytime_shannon_entropy", "energy_morning_std",
        "alterness_morning_std", "happiness_morning_median"
    ],
    "Evening Energy": [
        "energy_evening_median", "relax_evening_median",
        "health_evening_std", "energy_evening_std",
        "health_evening_median", "happiness_evening_median",
        "happiness_evening_std"
    ],
    "Caffeine Intake": [
        "caffeine_cups_shannon_entropy",
        "awake_type_shannon_entropy",
        "awake_type_std",
        "awake_type_median"
    ],
    "Sleep": [
        "sleep_efficiency_median",
        "sleep_duration_max",
        "nap_duration_fitbit_max"
    ]
}

In [6]:
data = pd.read_json("/content/all_participants_features.json")
data.head()

,1002,1102,1103,1105,1108,1115,1150,1151,1153,1154,...,1307,1308,1309,1310,1311,1312,1313,1314,1315,1316
Caffeine Intake,"{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.5, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 1.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 1.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...",...,"{'awake_type_median': 1.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon..."
Evening Energy,"{'energy_evening_median': 24.0, 'energy_evenin...","{'energy_evening_median': 41.0, 'energy_evenin...","{'energy_evening_median': 68.0, 'energy_evenin...","{'energy_evening_median': 80.0, 'energy_evenin...","{'energy_evening_median': 28.0, 'energy_evenin...","{'energy_evening_median': 70.5, 'energy_evenin...","{'energy_evening_median': 30.0, 'energy_evenin...","{'energy_evening_median': 35.0, 'energy_evenin...","{'energy_evening_median': 43.0, 'energy_evenin...","{'energy_evening_median': 31.0, 'energy_evenin...",...,"{'energy_evening_median': 16.0, 'energy_evenin...","{'energy_evening_median': 36.0, 'energy_evenin...","{'energy_evening_median': 0.0, 'energy_evening...","{'energy_evening_median': 30.0, 'energy_evenin...","{'energy_evening_median': 10.0, 'energy_evenin...","{'energy_evening_median': 12.5, 'energy_evenin...","{'energy_evening_median': 0.0, 'energy_evening...","{'energy_evening_median': 0.0, 'energy_evening...","{'energy_evening_median': 0.0, 'energy_evening...","{'energy_evening_median': 0.0, 'energy_evening..."
Heart Rate Variability,"{'RestingHeartRate_std': 3.201, 'hrentropy_std...","{'RestingHeartRate_std': 2.171, 'hrentropy_std...","{'RestingHeartRate_std': 1.956, 'hrentropy_std...","{'RestingHeartRate_std': 1.687, 'hrentropy_std...","{'RestingHeartRate_std': 2.103, 'hrentropy_std...","{'RestingHeartRate_std': 1.6520000000000001, '...","{'RestingHeartRate_std': 2.318, 'hrentropy_std...","{'RestingHeartRate_std': 1.324, 'hrentropy_std...","{'RestingHeartRate_std': 2.082, 'hrentropy_std...","{'RestingHeartRate_std': 1.974, 'hrentropy_std...",...,"{'RestingHeartRate_std': 2.659, 'hrentropy_std...","{'RestingHeartRate_std': 4.312, 'hrentropy_std...","{'RestingHeartRate_std': 3.65, 'hrentropy_std'...","{'RestingHeartRate_std': 3.229, 'hrentropy_std...","{'RestingHeartRate_std': 2.304, 'hrentropy_std...","{'RestingHeartRate_std': 1.867, 'hrentropy_std...","{'RestingHeartRate_std': 2.96, 'hrentropy_std'...","{'RestingHeartRate_std': 2.972, 'hrentropy_std...","{'RestingHeartRate_std': 3.002, 'hrentropy_std...","{'RestingHeartRate_std': 3.3, 'hrentropy_std':..."
Inactivity at Daytime,{'duration_entropy_non_step_std': 0.784},{'duration_entropy_non_step_std': 0.242},{'duration_entropy_non_step_std': 0.224},{'duration_entropy_non_step_std': 0.258},{'duration_entropy_non_step_std': 0.331},{'duration_entropy_non_step_std': 0.354},{'duration_entropy_non_step_std': 0.304},{'duration_entropy_non_step_std': 0.267},{'duration_entropy_non_step_std': 0.268},{'duration_entropy_non_step_std': 0.3460000000...,...,{'duration_entropy_non_step_std': 0.384},{'duration_entropy_non_step_std': 0.544},{'duration_entropy_non_step_std': 0.303},{'duration_entropy_non_step_std': 0.438},{'duration_entropy_non_step_std': 0.335},{'duration_entropy_non_step_std': 0.242},{'duration_entropy

In [7]:
detailed_thinking = "off"

def build_grouped_data(row):
    grouped = {}

    for group, columns in FEATURE_GROUPS.items():
        grouped[group] = {
            col: row[col]
            for col in columns
            if col in row and not pd.isna(row[col])
        }

    return grouped

In [8]:
def build_prompt(grouped_dict):
    formatted_sections = []

    for group, features in grouped_dict.items():
        section = f"## {group}\n"
        for k, v in features.items():
            section += f"{k}: {v}\n"
        formatted_sections.append(section)

    data_block = "\n".join(formatted_sections)

    prompt = f"""
You are a behavioral health data analyst.

Below is structured wearable and survey-derived feature data grouped by domain.

Write a concise professional summary describing:
- Circadian rhythm stability
- Cardiovascular variability
- Work/shift regularity
- Daytime inactivity
- Morning and evening energy patterns
- Sleep quality

Interpret patterns at a high level.
Do NOT list features individually.
Do NOT invent values.

Data:
{data_block}

Summary:
"""
    return prompt.strip()

In [9]:
def build_classification_prompt(summary):
    return f"""
You are a clinical burnout assessment assistant.

- Emotional Exhaustion: feeling emotionally drained, fatigued, or depleted.
- Depersonalization: detached, cynical, or impersonal attitudes toward others or work.
- Reduced Personal Accomplishment: feelings of incompetence or lack of achievement at work.

Based ONLY on the behavioral health summary provided,
- High
- Low

Rules:
- Output EXACTLY one word: High or Low
- Do not explain your reasoning
- Do not output anything else

Summary:
{summary}

Burnout Risk:
""".strip()

In [12]:
ground_truth = pd.read_csv("/content/jbs_scores_post.csv")
ground_truth.rename(columns={"u_id": "ID"}, inplace=True)
ground_truth["ID"] = ground_truth["ID"].astype(str)

In [13]:
!pkill -f vllm

**Orchestration Main Loop for Mental Health LLM Benchmarking**

In [16]:
NUM_ITERATIONS = 3

In [17]:
for model_key, config in models.items():
    hf_name = config["hf_name"]
    port = config["port"]
    gpu_mem = config["gpu_memory_utilization"]
    max_len = config["max_model_len"]

    print(f"\n{'='*50}")
    print(f"Starting pipeline for {model_key}")
    print(f"{'='*50}")

    # --- A. Git Clone the Model ---
    print("\n--- Preparing Model Weights ---")
    repo_dir = hf_name.split("/")[-1]
    if not os.path.exists(repo_dir):
        print(f"Starting git clone for {hf_name}...")
        try:
            subprocess.run(
                ["git", "clone", f"https://huggingface.co/{hf_name}"],
                check=True,
                capture_output=True,
                text=True
            )
            print(f"[SUCCESS] Git clone finished perfectly.")
        except subprocess.CalledProcessError as e:
            print(f"[FAILED] Git clone encountered an error:\n{e.stderr}")
            print(f"Skipping {model_key} and moving to the next model.")
            continue # Skip to the next model in the loop
    else:
        print(f"Directory '{repo_dir}' already exists. Skipping clone.")

    # --- B. Launch API Server ---
    print("\n---  Launching vLLM Server ---")
    log_file = f"vllm_{model_key}.log"
    cmd = [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", hf_name,
        "--trust-remote-code",
        "--host", "0.0.0.0",
        "--port", str(port),
        "--served-model-name", model_key,
        "--tensor-parallel-size", "1",
        "--max-model-len", str(max_len),
        "--gpu-memory-utilization", str(gpu_mem),
        "--enable-auto-tool-choice",
        "--tool-call-parser", "llama3_json"
    ]

    # Open log file and start process in the background
    with open(log_file, "w") as out:
        server_process = subprocess.Popen(cmd, stdout=out, stderr=out)

    server_ready = wait_for_server(log_file, server_process)
    if not server_ready:
        print(f"Skipping task execution for {model_key} due to server failure.")
        continue # Skip to next model

    # --- C. Run Classification Tasks ---
    print("\n---  Executing Inference ---")
    client = OpenAI(base_url=f"http://0.0.0.0:{port}/v1", api_key="dummy")

    run_accuracies = []

    for run in range(1, NUM_ITERATIONS + 1):
        print(f"\n--- Executing Inference (Run {run}/{NUM_ITERATIONS}) ---")

        # Reset lists for each run
        summary_results = []
        classification_results = []

        # Unique filenames for each run
        output_summary_file = f"llm_summaries_{model_key}_run{run}.json"
        with open(output_summary_file, "w") as f:
            json.dump([], f)

        def generate_summary(participant_id, grouped_data):
            try:
                resp = client.chat.completions.create(
                    model=model_key,
                    messages=[{"role": "user", "content": build_prompt(grouped_data)}],
                    temperature=0.3, # Variation happens here
                    extra_body={"detailed_thinking": "off"}
                )
                return {"status": "success", "data": {"ID": participant_id, "llm_summary": resp.choices[0].message.content.strip()}}
            except Exception as e:
                return {"status": "error", "participant_id": participant_id, "error": str(e)}

        def classify_participant(item):
            try:
                resp = client.chat.completions.create(
                    model=model_key,
                    messages=[{"role": "user", "content": build_classification_prompt(item["llm_summary"])}],
                    temperature=0.0,
                    max_tokens=3,
                    extra_body={"detailed_thinking": "off"}
                )
                return {"ID": item["ID"], "llm_summary": item["llm_summary"], "llm_burnout_label": resp.choices[0].message.content.strip()}
            except Exception as e:
                return None

        # Summarization Phase
        with concurrent.futures.ThreadPoolExecutor(max_workers=6) as executor:
            futures = [executor.submit(generate_summary, pid, gdata) for pid, gdata in data.items()]
            for future in concurrent.futures.as_completed(futures):
                result = future.result()
                if result and result["status"] == "success":
                    summary_results.append(result["data"])

            with open(output_summary_file, "w") as f:
                json.dump(summary_results, f, indent=4)

        # Classification Phase
        with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
            futures = [executor.submit(classify_participant, item) for item in summary_results]
            for future in concurrent.futures.as_completed(futures):
                res = future.result()
                if res:
                    classification_results.append(res)

        # Evaluation for this specific run
        df_results = pd.DataFrame(classification_results)
        if not df_results.empty:
            df_results.to_csv(f"llm_predictions_{model_key}_run{run}.csv", index=False)

            df_results["ID"] = df_results["ID"].astype(str)
            df_eval = df_results.merge(ground_truth, on="ID", how="inner")

            df_eval["llm_burnout_label"] = (
                df_eval["llm_burnout_label"].astype(str)
                .str.replace(r"[*]", "", regex=True)
                .str.strip().str.lower()
            )

            def standardize_label(val):
                if "high" in val: return "High"
                elif "low" in val: return "Low"
                return None

            df_eval["llm_burnout_label"] = df_eval["llm_burnout_label"].apply(standardize_label)
            if pd.api.types.is_numeric_dtype(df_eval["risk"]):
                df_eval["risk"] = df_eval["risk"].map({1: "High", 0: "Low"})

            acc = accuracy_score(df_eval["risk"], df_eval["llm_burnout_label"])
            run_accuracies.append(acc)
            print(f"Accuracy for Run {run}: {acc:.4f}")
        else:
            print(f"[WARNING] No results generated for Run {run}.")

    # average metrics
    if run_accuracies:
        avg_acc = np.mean(run_accuracies)
        std_acc = np.std(run_accuracies)
        print(f"\n{'*'*40}")
        print(f"FINAL METRICS FOR {model_key}:")
        print(f"Average Accuracy ({NUM_ITERATIONS} runs): {avg_acc:.4f} (±{std_acc:.4f})")
        print(f"{'*'*40}")
    else:
        print(f"\n[WARNING] Could not compute average metrics for {model_key}.")

    # --- E. Close the Server ---
    print("\n--- Shutting Down ---")
    print(f"Shutting down vLLM server for {model_key}...")
    server_process.terminate()
    server_process.wait()
    print(f"Server closed. Moving to next model.\n")

print("All models processed successfully!")


Starting pipeline for Llama-Nemotron-4B

--- Preparing Model Weights ---
Directory 'Llama-3.1-Nemotron-Nano-4B-v1.1' already exists. Skipping clone.

---  Launching vLLM Server ---
Waiting for vLLM server to spin up. Monitoring vllm_Llama-Nemotron-4B.log...
..............
[SUCCESS] Server is up and running! (Took 70.0 seconds)

---  Executing Inference ---

--- Executing Inference (Run 1/3) ---
Accuracy for Run 1: 0.7727

--- Executing Inference (Run 2/3) ---
Accuracy for Run 2: 0.7045

--- Executing Inference (Run 3/3) ---
Accuracy for Run 3: 0.8182

****************************************
FINAL METRICS FOR Llama-Nemotron-4B:
Average Accuracy (3 runs): 0.7652 (±0.0467)
****************************************

--- Shutting Down ---
Shutting down vLLM server for Llama-Nemotron-4B...
Server closed. Moving to next model.


Starting pipeline for BioMistral-7B

--- Preparing Model Weights ---
Directory 'BioMistral-7B' already exists. Skipping clone.

---  Launching vLLM Server ---
Waitin